In [1]:
import pandas as pd
import numpy as np
import os

data_path = os.path.join(os.path.expanduser("~"), "hr-projector", "data", "raw", "statcast_2022_2025.parquet")
df = pd.read_parquet(data_path)
print(f"Loaded {len(df):,} rows")

Loaded 2,868,714 rows


In [2]:
# filter for flyballs 
fb = df[df['bb_type'] == 'fly_ball'].copy()

# target 
fb['is_hr'] = (fb['events'] == 'home_run').astype(int)

print(f"Fly balls: {len(fb):,}")
print(f"Home runs: {fb['is_hr'].sum():,}")
print(f"HR rate: {fb['is_hr'].mean()*100:.1f}%")

Fly balls: 131,432
Home runs: 20,439
HR rate: 15.6%


In [3]:
# important features
feature_cols = [
    # Pitch characteristics
    'release_speed', 'release_spin_rate', 'pitch_type',
    # Contact quality
    'launch_speed', 'launch_angle',
    # Matchup
    'stand', 'p_throws',
    # Park
    'home_team',
    # Pitcher fatigue
    'pitcher_days_since_prev_game',
    # Game context
    'inning', 'balls', 'strikes',
]

target = 'is_hr'

model_df = fb[feature_cols + [target, 'game_date']].copy()

# Drop rows with nulls in critical columns
model_df = model_df.dropna(subset=['launch_speed', 'launch_angle', 'release_speed'])

# Fill minor nulls
model_df['release_spin_rate'] = model_df['release_spin_rate'].fillna(model_df['release_spin_rate'].median())
model_df['pitcher_days_since_prev_game'] = model_df['pitcher_days_since_prev_game'].fillna(5)

print(f"Clean rows: {len(model_df):,}")
print(f"HR rate after cleaning: {model_df['is_hr'].mean()*100:.1f}%")
print(model_df.dtypes)

Clean rows: 130,729
HR rate after cleaning: 15.6%
release_speed                          Float64
release_spin_rate                        Int64
pitch_type                              object
launch_speed                           Float64
launch_angle                             Int64
stand                                   object
p_throws                                object
home_team                               object
pitcher_days_since_prev_game             Int64
inning                                   Int64
balls                                    Int64
strikes                                  Int64
is_hr                                    int64
game_date                       datetime64[ns]
dtype: object


In [ ]:

model_df['stand_enc'] = (model_df['stand'] == 'R').astype(int)
model_df['p_throws_enc'] = (model_df['p_throws'] == 'R').astype(int)

# pitch types
pitch_dummies = pd.get_dummies(model_df['pitch_type'], prefix='pitch')
model_df = pd.concat([model_df, pitch_dummies], axis=1)

park_hr_rate = model_df.groupby('home_team')['is_hr'].mean().rename('park_hr_factor')
model_df = model_df.join(park_hr_rate, on='home_team')

print("Park HR factors:")
print(park_hr_rate.sort_values(ascending=False))

Park HR factors:
home_team
CIN    0.182585
ATL    0.181905
NYY    0.181102
LAD    0.179032
PHI    0.178638
LAA    0.177455
MIL    0.168476
TB     0.166412
NYM    0.165856
TOR    0.164020
COL    0.163209
CHC    0.158589
BOS    0.158121
WSH    0.156998
HOU    0.154856
BAL    0.153846
SEA    0.153440
TEX    0.151386
MIN    0.150170
SD     0.148951
MIA    0.146196
ATH    0.142920
AZ     0.142661
CWS    0.139576
DET    0.137915
CLE    0.136865
STL    0.136032
PIT    0.130998
SF     0.129337
KC     0.126917
Name: park_hr_factor, dtype: float64


In [6]:
final_features = [
    'launch_speed', 'launch_angle',
    'release_speed', 'release_spin_rate',
    'stand_enc', 'p_throws_enc',
    'pitcher_days_since_prev_game',
    'inning', 'balls', 'strikes',
    'park_hr_factor',
] + [col for col in model_df.columns if col.startswith('pitch_')]

# Drop the raw pitch_type string column from the feature list if it snuck in
final_features = [f for f in final_features if f != 'pitch_type']

X = model_df[final_features].astype(float)
y = model_df['is_hr']
dates = model_df['game_date']

print(f"Feature matrix shape: {X.shape}")
print(f"Features: {final_features}")

Feature matrix shape: (130729, 27)
Features: ['launch_speed', 'launch_angle', 'release_speed', 'release_spin_rate', 'stand_enc', 'p_throws_enc', 'pitcher_days_since_prev_game', 'inning', 'balls', 'strikes', 'park_hr_factor', 'pitch_CH', 'pitch_CS', 'pitch_CU', 'pitch_EP', 'pitch_FA', 'pitch_FC', 'pitch_FF', 'pitch_FO', 'pitch_FS', 'pitch_KC', 'pitch_KN', 'pitch_SC', 'pitch_SI', 'pitch_SL', 'pitch_ST', 'pitch_SV']


In [7]:
train_mask = dates.dt.year < 2025
test_mask = dates.dt.year == 2025

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

print(f"Train: {len(X_train):,} rows ({y_train.mean()*100:.1f}% HR rate)")
print(f"Test:  {len(X_test):,} rows ({y_test.mean()*100:.1f}% HR rate)")

Train: 97,482 rows (15.6% HR rate)
Test:  33,247 rows (15.6% HR rate)
